# 8. Train Cell-Trajectory JEPA

Phase A of the JEPA research trajectory: train a latent src→tgt predictor on LPS pairs.

- No MaskGIT demask/remask loop
- Loss = MSE in embedding space
- Warm-start token embeddings from a masking checkpoint
- Outputs on sod2; GPUs `5,6,7`

### Recommended: screen

```bash
screen -S jepa_train
bash /home/stuke1/perturbgen/Perturbgen/docs/examples/run_train_jepa_sod2.sh
```


In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/home/stuke1/perturbgen")
REPO = WORKSPACE / "Perturbgen"
TOKENIZED = WORKSPACE / "T_perturb" / "tokenized_data" / "LPS_all_tps_2k"
SOD2_ROOT = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")

os.environ["CUDA_VISIBLE_DEVICES"] = "5,6,7"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = str(SOD2_ROOT / "wandb")
os.environ["TMPDIR"] = str(SOD2_ROOT / "tmp")

OUTPUT_DIR = str(SOD2_ROOT / "T_perturb" / "res" / "jepa")
SRC_DATASET = str(TOKENIZED / "dataset_2000_hvg_src" / "normal.dataset")
TGT_DATASET_FOLDER = str(TOKENIZED / "dataset_2000_hvg_tgt")
SRC_ADATA = str(TOKENIZED / "h5ad_pairing_2000_hvg_src" / "normal.h5ad")
TGT_ADATA_FOLDER = str(TOKENIZED / "h5ad_pairing_2000_hvg_tgt")
MAPPING_DICT_PATH = str(TOKENIZED / "token_id_to_genename_2000_hvg.pkl")
TOKENID_TO_ROWID = str(TOKENIZED / "tokenid_to_rowid_2000_hvg.pkl")
ENCODER_PATH = str(
    REPO / "pretraining_cohort"
    / "20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"
)
CKPT_MASKING = str(
    WORKSPACE / "T_perturb" / "res" / "masking" / "checkpoints"
    / "20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt"
)
print("OUTPUT_DIR", OUTPUT_DIR)
print("CKPT_MASKING exists", Path(CKPT_MASKING).is_file())


In [ ]:
cmd = [
    "python", "-m", "perturbgen", "train-jepa",
    "--train_mode", "jepa",
    "--split", "False",
    "--output_dir", OUTPUT_DIR,
    "--log_dir", str(SOD2_ROOT / "logs"),
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--tokenid_to_rowid_path", TOKENID_TO_ROWID,
    "--encoder_path", ENCODER_PATH,
    "--ckpt_masking_path", CKPT_MASKING,
    "--batch_size", "64",
    "--epochs", "20",
    "--cellgen_lr", "1e-4",
    "--cellgen_wd", "1e-4",
    "--num_layers", "2",
    "--d_ff", "1024",
    "--d_model", "768",
    "--pred_tps", "1", "2", "3",
    "--var_list", "cell_type_harmonized", "time_after_LPS",
    "--ema_decay", "0.996",
    "--normalize_latents", "true",
    "--jepa_loss", "mse",
    "--wandb_mode", "offline",
    "--seed", "0",
]
print(" ".join(cmd))


## Phases B–F

After `JEPATrainer.test` writes embeddings:

```bash
python -m perturbgen eval-jepa --phase all \
  --jepa_embeddings $OUTPUT_DIR/embeddings/jepa_cell_embeddings.pt \
  --output_dir $OUTPUT_DIR/eval
```

Phase D decoder training: `python -m perturbgen train-jepa-decoder ... --ckpt_jepa_path <jepa.ckpt>`
